# Indonesian Reddit sentiment labeling on GPU

This notebook runs the language-routed Approach 2 pipeline. It is designed for Google Colab or Kaggle GPU notebooks and supports exactly 10,000 comments or the full dataset.

**Approach 2:** Indonesian comments → `w11wo/indonesian-roberta-base-sentiment-classifier`; English comments → `cardiffnlp/twitter-roberta-base-sentiment-latest`. Unknown/short/code-mixed comments route to the Indonesian model.

The output stores canonical probabilities (`p_negative`, `p_neutral`, `p_positive`) as well as the selected label and confidence.

## 1. Enable a GPU

- **Colab:** Runtime → Change runtime type → T4 GPU.
- **Kaggle:** Notebook options → Accelerator → GPU T4/P100.

Upload the project files or attach a Kaggle Dataset containing `data/cache/comments.parquet`, `data/cache/comments_text.parquet`, `src/sentiment_gpu.py`, and optionally `data/derived/gold_llm.csv`.

In [ ]:
# Run this in Colab/Kaggle after uploading or attaching the project.
# Change PROJECT to the repository root (the folder that contains src/ and data/).
PROJECT = '/content/indonesian-subreddit-dataset'  # Colab example
# PROJECT = '/kaggle/input/indonesian-subreddit-dataset'  # Kaggle example

%cd $PROJECT
!pip -q install -U -r requirements-gpu.txt
!nvidia-smi

In [ ]:
# Phase 1: exactly 10,000 unique comments.
!python src/sentiment_gpu.py \
  --input-dir data/cache \
  --limit 10000 \
  --seed 42 \
  --output data/derived/sentiment_10k.parquet \
  --batch-size 64 \
  --chunk-size 10000

In [ ]:
# Inspect the result.
import pandas as pd, json
pred = pd.read_parquet('data/derived/sentiment_10k.parquet')
print(pred.shape)
print('unique IDs:', pred.comment_id.nunique())
print(pred.label.value_counts())
print(pred.route.value_counts())
print(pred.confidence.describe())
display(pred.head())

In [ ]:
# Optional: benchmark against the independent LLM-labeled gold file.
!python src/benchmark_sentiment.py \
  --predictions data/derived/sentiment_10k.parquet \
  --gold data/derived/gold_llm.csv

## 2. Full dataset run

After checking the 10K output, classify all unique non-empty comments. A T4 GPU should be dramatically faster than the local CPU run. The script checkpoints every 10,000 rows and resumes from `sentiment_full.checkpoint.parquet` if interrupted.

```bash
python src/sentiment_gpu.py --input-dir data/cache --limit 728522 --output data/derived/sentiment_full.parquet --batch-size 64 --chunk-size 10000
```

Copy the resulting Parquet file back into `data/derived/sentiment_full.parquet`, then rebuild the dashboard artifacts locally with `python src/build_sentiment_cache.py`.